# 🚀 Bank Marketing Subscription Prediction - Google Colab Quickstart
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Akshar701/bank-marketing-ann/blob/main/notebooks/colab_quickstart.ipynb)

This interactive Colab notebook allows teammates to collaborate, test pretrained models, run customer predictions, and experiment with Deep Artificial Neural Networks (ANN) & classical ML benchmarks directly in the browser with **zero local setup**.

### Step 1: Environment Setup & Clone Repository
Clones the repository, installs required dependencies, and sets the Keras 3 backend to PyTorch.

In [ ]:
import os
import sys

# Detect Google Colab environment
IN_COLAB = 'google.colab' in str(get_ipython()) if 'get_ipython' in globals() else False

if IN_COLAB:
    print("🚀 Google Colab environment detected. Cloning repository...")
    if not os.path.exists("bank-marketing-ann"):
        !git clone https://github.com/Akshar701/bank-marketing-ann.git
    %cd bank-marketing-ann
    !pip install -q -r requirements.txt
    print("✅ Installed dependencies!")

# Ensure Keras uses PyTorch backend
os.environ['KERAS_BACKEND'] = 'torch'

# Ensure project root is in sys.path
PROJECT_ROOT = os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Working Directory:", os.getcwd())


### Step 2: Load Dataset & Verify Schema
Loads `bank-full.csv` (45,211 rows) and inspects the class distribution.

In [ ]:
import pandas as pd
from src.data_loader import load_raw_data, validate_data_schema

data_path = os.path.join(PROJECT_ROOT, "data/bank-full.csv")
df = load_raw_data(data_path, sep=";")
print(f"Dataset Shape: {df.shape[0]:,} rows x {df.shape[1]} columns")
print("Target Distribution:")
display(df['y'].value_counts(normalize=True).map('{:.2%}'.format))
df.head(3)


### Step 3: Instant Interactive Inference with `BankPredictor`
Run inference using our best-performing **Advanced ANN** (PR-AUC: 0.6004) and **Pre-call XGBoost**.

In [ ]:
from src.predict import BankPredictor

cfg_path = os.path.join(PROJECT_ROOT, "config/config.yaml")
predictor = BankPredictor(config_path=cfg_path)

# Sample customer profile for telemarketing scoring
customer = {
    'age': 38,
    'job': 'technician',
    'marital': 'single',
    'education': 'tertiary',
    'default': 'no',
    'balance': 2400,
    'housing': 'no',
    'loan': 'no',
    'contact': 'cellular',
    'day': 15,
    'month': 'aug',
    'duration': 350,
    'campaign': 1,
    'pdays': 120,
    'previous': 3,
    'poutcome': 'success'
}

result = predictor.predict(customer, model_name="Advanced ANN", mode="post_call")
print("🎯 Post-Call Advanced ANN Prediction:")
for k, v in result.items():
    print(f"  • {k}: {v}")


### Step 4: Pre-call vs Post-call (Target Leakage Demonstration)
Compare decisions made **before dialing** (without knowing call duration) versus **after dialing**.

In [ ]:
pre_customer = dict(customer)
del pre_customer['duration']

pre_result = predictor.predict(pre_customer, model_name="Pre-call XGBoost (No Duration)", mode="pre_call")
print("📞 Pre-Call Lead Scoring (No Duration Leakage):")
for k, v in pre_result.items():
    print(f"  • {k}: {v}")


### Step 5: Benchmark Evaluation Table
Inspect the test metrics across all 9 benchmarked architectures ($N=6,782$ holdout test partition).

In [ ]:
comp_path = os.path.join(PROJECT_ROOT, "reports/model_comparison.csv")
comp_df = pd.read_csv(comp_path)
display(comp_df[['Model', 'Type', 'PR-AUC', 'ROC-AUC', 'F1-Score (0.5)', 'F1-Score (Opt Thresh)', 'Recall (Opt Thresh)', 'Brier Score']])


### Step 6: Visualizing Benchmark Charts
Displays the ROC curves, PR curves, and explainability plots.

In [ ]:
from IPython.display import Image, display

fig_dir = os.path.join(PROJECT_ROOT, "reports/figures")
charts = [
    "roc_curves_comparison.png",
    "pr_curves_comparison.png",
    "shap_summary_plot.png",
    "duration_leakage_analysis.png"
]

for chart in charts:
    chart_path = os.path.join(fig_dir, chart)
    if os.path.exists(chart_path):
        print(f"\n📈 {chart}:")
        display(Image(filename=chart_path))
